In [ ]:
"""
Grouped-Query Attention uses fewer keys and values than the number of query heads. It comes
from the 2023 paper (https://arxiv.org/pdf/2305.13245) and positions itself in between full 
MHA and multi-query attention (MQA where a single key and value is shared across all heads).
"""

In [1]:
import math
import torch
from torch import nn
from torch.nn import functional as F

class GQA(nn.Module):
    def __init__(self, d_emb, n_groups, n_kvheads):
        """
        n_groups: explains how many query groups exist and n_heads how many kv heads there are.
        If n_kvheads == n_groups -> MHA
        if n_kvheads == 1 -> MQA
        I think I should keep the query dimension as d_emb // n_groups and scale kv_head_dim down
        """
        super().__init__()
        self.n_groups = n_groups
        self.n_kvheads = n_kvheads
        self.d_emb = d_emb
        self.group_dim = d_emb // n_groups

        assert d_emb % n_groups == 0
        assert n_groups % n_kvheads == 0
        self.gh_ratio = n_groups // n_kvheads # group-to-head ratio: how many groups are there for each head

        self.o_proj = nn.Linear(d_emb, d_emb)
        # if we have 2 groups per kv head our group-to-head ratio is 2
        self.k_proj = nn.Linear(d_emb, self.n_kvheads * self.group_dim)
        self.v_proj = nn.Linear(d_emb, self.n_kvheads * self.group_dim)
        self.q_proj = nn.Linear(d_emb, d_emb)


    def split_heads_broadcastable(self, x):
        # x: B, S, D
        # x_int: B, S, kv, (1 or gh_ratio), group_dim
        # out: B, (1 or gh_ratio), kv, S, group_dim
        x = x.view(x.shape[0], x.shape[1], self.n_kvheads, -1, self.group_dim)
        return x.transpose(1,3)

    def merge_heads(self, x):
        # x: [B, gh_ratio, kv_heads, S, group_dim]
        # out: B, S, D
        B, _, _, S, _ = x.shape
        # att: [B, S, kv_heads, gh_ratio, group_dim]
        x = x.transpose(1, 3).contiguous()
        # att: [B, S, D]
        return x.view(B, S, self.d_emb)
    

    def forward(self, x, causal=False):
        # x [B, S, D]
        # k [B, S, D / gh_ratio]
        k = self.k_proj(x)
        v = self.v_proj(x)
        q = self.q_proj(x)
        print("k: ", k.shape)
        print("q: ", q.shape)

        # kv [B, 1, kvheads, S, group_dim]
        k = self.split_heads_broadcastable(k)
        v = self.split_heads_broadcastable(v)
        print("k: ", k.shape)

        # q [B, gh_ratio, kvheads, S, group_dim]
        q = self.split_heads_broadcastable(q)
        print("q: ", q.shape)

        # att: softmax(q @ k.T / sqrt(group_dim)) @ v    

        # [B, gh_ratio, kv_heads, S, group_dim] @ [B, 1, kv_heads, group_dim, S] = [B, gh_ratio, kv_heads, S, S]
        scores = q @ k.transpose(-1, -2) / math.sqrt(self.group_dim)

        if causal:
            mask = torch.ones((scores.shape[-2], scores.shape[-1]), device=scores.device, dtype=torch.bool).tril(diagonal=0)
            scores = scores.masked_fill(~mask, float("-inf"))

        scores = F.softmax(scores, dim=-1)
        # [B, gh_ratio, kv_heads, S, S] @ [B, 1, kv_heads, S, group_dim] = [B, gh_ratio, kv_heads, S, group_dim]
        att = scores @ v

        att = self.merge_heads(att)

        return self.o_proj(att)

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"

gqa = GQA(16, 4, 2).to(device)

# B=1, S=5, d_emb=16
x = torch.randn(1, 5, 16, device=device)
self_attention = gqa(x, causal=True)
self_attention.shape

k:  torch.Size([1, 5, 8])
q:  torch.Size([1, 5, 16])
k:  torch.Size([1, 1, 2, 5, 4])
q:  torch.Size([1, 2, 2, 5, 4])


torch.Size([1, 5, 16])